# EU Climate Policy RAG

A simple RAG baseline for answering questions about EU climate policy, including the European Climate Law, the European Green Deal, the Fit for 55 package, the European Union Emissions Trading System, the Carbon Border Adjustment Mechanism, and the Paris Agreement. It uses cleaned passages from `data/eu_climate_policy.json`, builds a local `minsearch` index, and chains the core steps: `search`, `build_prompt`, `llm`, and `rag`.

In [1]:
import json

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from minsearch import Index

openai_client = OpenAI()

## 1. Load Your Data

Curated passages from official EU climate policy documents stored in `data/eu_climate_policy.json`. Each document has a `source`, `article`, `topic`, and `text` field.

In [2]:
with open("../data/eu_climate_policy.json") as f:
    documents = json.load(f)

print(f"Loaded {len(documents)} documents")
print("Sample:", documents[0])

Loaded 14 documents
Sample: {'source': 'COM 2023 796 final EU wide assessment draft updated National Energy and Climate Plans', 'topic': 'energy_and_climate', 'file_path': 'climate_policy_docs/COM_2023_796_final_EU_wide_assessment_draft_updated_National_Energy_and_Climate_Plans.md', 'article': 'document', 'text': "![european flag](./../../../../images/eclogo.jpg)EUROPEAN COMMISSION\n\nBrussels, 18.12.2023\n\nCOM(2023) 796 final\n\nCOMMUNICATION FROM THE COMMISSION TO THE EUROPEAN PARLIAMENT, THE COUNCIL, THE EUROPEAN ECONOMIC AND SOCIAL COMMITTEE AND THE COMMITTEE OF THE REGIONS\n\nEU wide assessment of the draft updated National Energy and Climate Plans\n\nAn important step towards the more ambitious 2030 energy and climate objectives under the European Green Deal and RePowerEU\n\n1INTRODUCTION - UPDATED INTEGRATED NATIONAL ENERGY AND CLIMATE PLANS: AN IMPORTANT STEP TOWARDS DELIVERING ON THE GREEN DEAL AND THE REPOWEREU PLAN\n\nRecent years have clearly shown the need for the EU to k

## 2. Build the Index

minsearch builds an in-memory TF-IDF index over the fields you specify.

In [3]:
index = Index(
    text_fields=["text", "source", "topic"],
    keyword_fields=["article"],
)

index.fit(documents)

## 3. Define the RAG Functions

`search` retrieves relevant documents, `build_prompt` formats them into a prompt, `llm` calls the model, and `rag` chains all three.

In [4]:
def search(query):
    return index.search(query, num_results=5)

In [5]:
instructions = """
You are an expert assistant on EU climate policy. Answer the QUESTION using only
the information in the CONTEXT below, which contains passages from official EU
documents (European Climate Law, European Green Deal, Fit for 55, EU ETS, CBAM,
and the Paris Agreement).

Rules:
- Cite the source and article for every claim you make (e.g. "European Climate Law, Article 4").
- If the context does not contain enough information to answer, say so explicitly -  do not guess.
- Be concise and accurate. Connect related targets or policies when the context supports it.
""".strip()


def build_prompt(query, search_results):
    context_items = []
    for doc in search_results:
        context_items.append(
            f"[{doc['source']} | {doc['article']} | topic: {doc['topic']}]\n{doc['text']}"
        )
    context_text = "\n\n".join(context_items)

    user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{context_text}
</CONTEXT>
""".strip()

    return user_prompt

In [6]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [7]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## 4. Try It Out

Ask a question a real student or policy learner would ask.

In [8]:
print(rag("How does the EU's 2030 climate target relate to the 2050 goal?"))

The EU's 2030 climate target, which mandates a reduction of net greenhouse gas emissions by at least 55% compared to 1990 levels, is a critical milestone on the path to achieving the 2050 climate-neutrality goal. The European Climate Law establishes this framework, ensuring that the 2030 target aligns with the long-term objective of becoming climate-neutral by 2050 (European Climate Law, Article 2 and Article 4).

In this context, the 2030 target is designed to set an interim benchmark that facilitates the transition towards the more ambitious 2050 goal, effectively providing a clear trajectory for emissions reductions across all sectors (European Climate Law, Article 4). The legally binding nature of both targets helps to enhance certainty for investors and stakeholders, promoting a cohesive approach to achieving climate objectives across the EU (European Climate Law, Article 1).

By establishing these linked targets, the European Union aims to ensure that each step taken towards the 